# **Lab 9 - Visión Por Computadora**

Integrantes:

- Francis Aguilar, 22243
- César López, 22535
- Jose Marchena, 22398

Repo:
https://github.com/MarchMol/lab9_vision


Usted ha sido contratado como Ingeniero de Visión por Computadora (CV Engineer) principal para la convención de anime y tecnología más grande de su ciudad: el Akihabara Fest. La organización requiere automatizar la logística y seguridad del evento utilizando cámaras de circuito cerrado (CCTV) y dispositivos Edge (Raspberry Pi 5 y NVIDIA Jetson Nano). Tendrá que tomar decisiones arquitectónicas críticas, resolver problemas de oclusión en multitudes de cosplayers y escribir un pipeline de inferencia desde cero


## Task 1 - Investigación

Además de las cámaras de seguridad con YOLO, los organizadores del Akihabara Fest quieren instalar "Cabinas Fotográficas de Realidad Aumentada (AR)" para los asistentes VIP. El objetivo de estas cabinas es doble:

1. Eliminar automáticamente el fondo real de la convención y reemplazarlo por escenarios de anime (ej. La Aldea Oculta de la Hoja de Naruto o Neo-Tokyo de Akira), simulando un "Chroma Key" (pantalla verde) perfecto sin usar tela verde.

2. Identificar específicamente la espada o báculo mágico que sostiene el usuario para superponerle un efecto de "brillo de energía" digital, diferenciándolo de las armas de otros amigos que estén en la misma foto.

La mesa directiva no sabe qué modelo de IA usar y le pide a usted investigar dos arquitecturas famosas: UNet y Mask R-CNN. Entonces, realice una investigación en artículos o documentación técnica oficial y redacte un reporte ejecutivo (máximo 2 página) respondiendo con rigor técnico a lo siguiente:

1. Diferencia Fundamental: Defina con sus propias palabras la diferencia exacta a nivel de píxel entre la Segmentación Semántica y la Segmentación de Instancia.

2. El Caso U-Net: Explique por qué la arquitectura U-Net (Segmentación Semántica) es ideal y altamente eficiente para la tarea 1 (separar a todos los humanos del fondo), pero explique técnicamente por qué fracasaría si intentamos usarla para la tarea 2 si hay dos réplicas de espadas idénticas cruzadas en la foto

3. El Caso Mask R-CNN: Explique cómo la arquitectura Mask R-CNN (Segmentación de Instancia) resuelve el problema anterior basándose en su naturaleza de "dos etapas" (recordando que hereda de Faster R-CNN). ¿Por qué Mask R-CNN sí podría iluminar una espada de rojo y otra de azul, incluso si se están tocando en la imagen?


## Task 2

En el pabellón principal del evento, se está llevando a cabo un concurso de cosplay. De repente, un grupo de 15 personas haciendo cosplay de Naruto entran al escenario exactamente iguales y posan muy juntos (simulando el Kage Bunshin no Jutsu o Jutsu Clones de Sombra). Las cámaras registran a los clones abrazados, superpuestos y parcialmente ocluidos unos detrás de otros.

Usted nota que su modelo clásico basado en YOLOv8 falla rotundamente: la red neuronal sí encuentra a los 15 clones y dibuja cientos de cajas, pero al final en la pantalla solo aparecen 4 o 5 personas detectadas. El resto "desaparece". Considerando esto, responda:


1. Explique matemáticamente, utilizando la fórmula del IoU (Intersección sobre Unión), por qué el algoritmo NMS está causando que los clones superpuestos desaparezcan (Falsos Negativos).


R//El algoritmo NMS elimina cajas redundantes usando el IoU

$$IoU = \frac{|B_1 \cap B_2|}{|B_1 \cup B_2|}$$

Donde $B_1$, $B_2$ son dos bounding boxes, el numerador es la área de intersección y denominador es el área de unión.

En este caso los 15 clones están muy juntos y superpuestos, las cajas predichas para diferentes personas tienen alto solapamiento
Entonces:

$$IoU(B_1, B_j) \approx 0.6 - 0.9$$

NMS ordena las cajas por confidence score, toma la mejor cadja y elimina todas las demas que cumplan:

$$IoU > threshold$$

Entonces el resultado es de muchas cajas validas (que son personas distintas) tienen IoU alto, NMS las considera duplicadas y las elimina como falsos negativos.


2. Si usted es el ingeniero a cargo y solo puede modificar los hiperparámetros en el código durante la inferencia: ¿Qué pasaría si ajusta el umbral de IoU del NMS a 0.15? ¿Qué pasaría si lo ajusta a 0.95? Justifique qué valor sería más adecuado para este problema de alta densidad.


R//
Caso IoU = 0.15 como muy bajo:
Casi cualquier solapamiento elimina cajas, e incluso pequeñas intersecciones se consideran duplicados, entonces se eliminan demasiadas cajas, muchisimos falsos negativos aparecen y se detectan aun menos personas.

Caso IoU = 0.95 como muy alto:
Sola se eliminan cajas casi identicas y hay cajas superpuestas pero diferentes sobreviven. Aqui se conservan mas personas, aparecen mas cajas duplicadas que son consideradas ruidos y aparecen tambien mas falsos positivos.

Entonces para escenas de alta densidad / oclusión, ya que esto permite distinguir personas cercas, evita eliminar detecciones válidas y reduce falsos negativos.


3. Si el presupuesto le permitiera cambiar el modelo a YOLOv10, explique cómo su arquitectura Asignación Dual de Etiquetas (Dual Label Assignment) resolvería este problema de oclusión sin necesidad de tocar el NMS.


R// YOLOv10 soluciona este problema usando DLA, que básicamente ayuda al modelo a entender mejor cuando hay muchas personas juntas y superpuestas. A diferencia de YOLOv8, donde varias personas compiten por la misma zona y terminan generando cajas muy parecidas que luego el NMS elimina, YOLOv10 entrena al modelo para ver un mismo objeto desde varias “perspectivas”, lo que hace que no pierda detecciones en escenas densas.


## Task 3

El cliente quedó fascinado con sus propuestas teóricas y ahora quiere un Prototipo Funcional (MVP). Desean una especie de "Pokedex" en tiempo real usando la cámara web de una computadora portátil para escanear y clasificar objetos en la convención.

Usted debe escribir un script de Python desde cero utilizando la librería ultralytics y OpenCV. No es necesario entrenar un modelo personalizado; puede utilizar los pesos pre-entrenados estándar (COCO dataset) o descargar un modelo pre-entrenado de la comunidad (ej. detección de caras de anime o cartas TCG), pero el código de inferencia debe ser suyo. El script que realice debe tener:

1. Captura de Video: El programa debe abrir la cámara web (o leer un video mp4 pregrabado de temática geek que usted proporcione) utilizando OpenCV (cv2.VideoCapture).

2. Inferencia Continua: Instanciar un modelo de la familia YOLO (usted elige la versión, pero debe comentarlo en el código). Procesar cada frame capturado.

3. Manipulación de Hiperparámetros: El código debe pasar explícitamente como argumentos los valores de conf (Confidence threshold) y iou (NMS IoU threshold) a la función de inferencia del modelo.

4. Cálculo de FPS: Usted debe calcular algorítmicamente y en tiempo real los FPS (Frames Per Second) de la inferencia utilizando la librería time, y sobreponer ese número en la esquina del video final.

5. Extracción de Tensores: Para cada detección, extraiga las coordenadas `(𝑥1, 𝑦1, 𝑥2, 𝑦2)` del tensor de salida y la etiqueta (clase), y dibújelas usando funciones primitivas de OpenCV (cv2.rectangle, cv2.putText), no usando el método .plot() automático de la librería. Esto demuestra que usted sabe iterar sobre el tensor de resultados.

Se espera que el entregable de este laboratorio sea un video de 5 segundos o un gif mostrando la ejecución existosa del programa, con los FPS visibles y las cjaas delimitadoras dibujadas.
